In [1]:
import pandas as pd

fund_master = pd.read_csv('../data/raw/01_fund_master.csv')
nav_history = pd.read_csv('../data/raw/02_nav_history.csv')import requests
import pandas as pd

# Fetch data from mfapi.in
url = "https://api.mfapi.in/mf/125497"
response = requests.get(url)
response.raise_for_status()  # raises an error if the request failed

data = response.json()

# Inspect the structure first
print(data['meta'])       # fund metadata (name, house, etc.)
print(data['data'][:5])   # first 5 NAV records

{'fund_house': 'SBI Mutual Fund', 'scheme_type': 'Open Ended Schemes', 'scheme_category': 'Equity Scheme - Small Cap Fund', 'scheme_code': 125497, 'scheme_name': 'SBI Small Cap Fund - Direct Plan - Growth', 'isin_growth': 'INF200K01T51', 'isin_div_reinvestment': None}
[{'date': '24-07-2026', 'nav': '204.85350'}, {'date': '23-07-2026', 'nav': '205.55390'}, {'date': '22-07-2026', 'nav': '207.67480'}, {'date': '21-07-2026', 'nav': '210.09000'}, {'date': '20-07-2026', 'nav': '209.44660'}]


In [2]:
# Convert the NAV history into a DataFrame
nav_df = pd.DataFrame(data['data'])

# Add fund identifying info from meta
nav_df['scheme_code'] = data['meta']['scheme_code']
nav_df['scheme_name'] = data['meta']['scheme_name']
nav_df['fund_house'] = data['meta']['fund_house']

print(nav_df.shape)
print(nav_df.head())
print(nav_df.dtypes)

(3129, 5)
         date        nav  scheme_code  \
0  24-07-2026  204.85350       125497   
1  23-07-2026  205.55390       125497   
2  22-07-2026  207.67480       125497   
3  21-07-2026  210.09000       125497   
4  20-07-2026  209.44660       125497   

                                 scheme_name       fund_house  
0  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
1  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
2  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
3  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
4  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
date             str
nav              str
scheme_code    int64
scheme_name      str
fund_house       str
dtype: object


In [3]:
output_path = '../data/raw/hdfc_top100_direct_nav_live.csv'
nav_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to ../data/raw/hdfc_top100_direct_nav_live.csv


In [1]:
import requests
import pandas as pd
import time

# Scheme codes to fetch
schemes = {
    "119551": "SBI Bluechip",
    "120503": "ICICI Bluechip",
    "118632": "Nippon Large Cap",
    "119092": "Axis Bluechip",
    "120841": "Kotak Bluechip",
}

all_dfs = []

for code, name in schemes.items():
    print(f"Fetching {name} ({code})...")
    url = f"https://api.mfapi.in/mf/{code}"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data['data'])
    df['scheme_code'] = data['meta']['scheme_code']
    df['scheme_name'] = data['meta']['scheme_name']
    df['fund_house'] = data['meta']['fund_house']

    all_dfs.append(df)
    print(f"  -> {df.shape[0]} NAV records fetched")

    time.sleep(1)  # small delay to be polite to the API

print("\nDone fetching all 5 schemes.")

Fetching SBI Bluechip (119551)...
  -> 3274 NAV records fetched
Fetching ICICI Bluechip (120503)...
  -> 3345 NAV records fetched
Fetching Nippon Large Cap (118632)...
  -> 3336 NAV records fetched
Fetching Axis Bluechip (119092)...
  -> 3603 NAV records fetched
Fetching Kotak Bluechip (120841)...
  -> 3339 NAV records fetched

Done fetching all 5 schemes.


In [2]:
combined_df = pd.concat(all_dfs, ignore_index=True)

print(combined_df.shape)
print(combined_df['scheme_name'].value_counts())  # confirm all 5 funds present
print(combined_df.head())

(16897, 5)
scheme_name
HDFC Money Market Fund - Growth Option - Direct Plan                     3603
Axis ELSS Tax Saver Fund - Direct Plan - Growth Option                   3345
quant Mid Cap Fund - Growth Option - Direct Plan                         3339
Nippon India Large Cap Fund - Direct Plan Growth Plan - Growth Option    3336
Aditya Birla Sun Life Banking & PSU Debt Fund  - DIRECT - IDCW           3274
Name: count, dtype: int64
         date        nav  scheme_code  \
0  24-07-2026  106.59610       119551   
1  23-07-2026  106.58510       119551   
2  22-07-2026  106.61070       119551   
3  21-07-2026  106.65670       119551   
4  20-07-2026  106.57370       119551   

                                         scheme_name  \
0  Aditya Birla Sun Life Banking & PSU Debt Fund ...   
1  Aditya Birla Sun Life Banking & PSU Debt Fund ...   
2  Aditya Birla Sun Life Banking & PSU Debt Fund ...   
3  Aditya Birla Sun Life Banking & PSU Debt Fund ...   
4  Aditya Birla Sun Life Banking &

In [3]:
output_path = '../data/raw/key_schemes_nav_live.csv'
combined_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to ../data/raw/key_schemes_nav_live.csv


In [1]:
import pandas as pd

fund_master = pd.read_csv('../data/raw/01_fund_master.csv')

print("Shape:", fund_master.shape)
print("\nColumns:", fund_master.columns.tolist())

Shape: (40, 15)

Columns: ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']


In [2]:
print("=" * 60)
print("UNIQUE FUND HOUSES")
print("=" * 60)
print(fund_master['fund_house'].unique())
print(f"Total: {fund_master['fund_house'].nunique()}")

print("\n" + "=" * 60)
print("UNIQUE CATEGORIES")
print("=" * 60)
print(fund_master['category'].unique())

print("\n" + "=" * 60)
print("UNIQUE SUB-CATEGORIES")
print("=" * 60)
print(fund_master['sub_category'].unique())

print("\n" + "=" * 60)
print("UNIQUE RISK GRADES")
print("=" * 60)
print(fund_master['risk_grade'].unique())

UNIQUE FUND HOUSES
<StringArray>
[         'SBI Mutual Fund',         'HDFC Mutual Fund',
      'ICICI Prudential MF',          'Nippon India MF',
        'Kotak Mahindra MF',         'Axis Mutual Fund',
 'Aditya Birla Sun Life MF',          'UTI Mutual Fund',
           'Mirae Asset MF',          'DSP Mutual Fund']
Length: 10, dtype: str
Total: 10

UNIQUE CATEGORIES
<StringArray>
['Equity', 'Debt']
Length: 2, dtype: str

UNIQUE SUB-CATEGORIES
<StringArray>
[      'Large Cap',       'Small Cap',            'Gilt',         'Mid Cap',
  'Short Duration',           'Value',          'Liquid',       'Index/ETF',
       'Flexi Cap',           'Index', 'Large & Mid Cap',            'ELSS']
Length: 12, dtype: str

UNIQUE RISK GRADES


KeyError: 'risk_grade'

In [3]:
print("Fund houses by fund count:")
print(fund_master['fund_house'].value_counts())

print("\nCategory breakdown:")
print(fund_master['category'].value_counts())

print("\nSub-category breakdown:")
print(fund_master['sub_category'].value_counts())

print("\nRisk grade breakdown:")
print(fund_master['risk_grade'].value_counts())

Fund houses by fund count:
fund_house
SBI Mutual Fund             5
HDFC Mutual Fund            5
ICICI Prudential MF         5
Nippon India MF             5
Kotak Mahindra MF           4
Axis Mutual Fund            4
Aditya Birla Sun Life MF    3
UTI Mutual Fund             3
Mirae Asset MF              3
DSP Mutual Fund             3
Name: count, dtype: int64

Category breakdown:
category
Equity    34
Debt       6
Name: count, dtype: int64

Sub-category breakdown:
sub_category
Large Cap          14
Mid Cap             7
Small Cap           6
Liquid              3
Gilt                2
Flexi Cap           2
Short Duration      1
Value               1
Index/ETF           1
Index               1
Large & Mid Cap     1
ELSS                1
Name: count, dtype: int64

Risk grade breakdown:


KeyError: 'risk_grade'

In [1]:
import pandas as pd

fund_master = pd.read_csv('../data/raw/01_fund_master.csv')

print("Shape:", fund_master.shape)
print("\nColumns:", fund_master.columns.tolist())

Shape: (40, 15)

Columns: ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']


In [2]:
print("=" * 60)
print("UNIQUE FUND HOUSES")
print("=" * 60)
print(fund_master['fund_house'].unique())
print(f"Total: {fund_master['fund_house'].nunique()}")

print("\n" + "=" * 60)
print("UNIQUE CATEGORIES")
print("=" * 60)
print(fund_master['category'].unique())

print("\n" + "=" * 60)
print("UNIQUE SUB-CATEGORIES")
print("=" * 60)
print(fund_master['sub_category'].unique())

print("\n" + "=" * 60)
print("UNIQUE RISK GRADES")
print("=" * 60)
print(fund_master['risk_grade'].unique())

UNIQUE FUND HOUSES
<StringArray>
[         'SBI Mutual Fund',         'HDFC Mutual Fund',
      'ICICI Prudential MF',          'Nippon India MF',
        'Kotak Mahindra MF',         'Axis Mutual Fund',
 'Aditya Birla Sun Life MF',          'UTI Mutual Fund',
           'Mirae Asset MF',          'DSP Mutual Fund']
Length: 10, dtype: str
Total: 10

UNIQUE CATEGORIES
<StringArray>
['Equity', 'Debt']
Length: 2, dtype: str

UNIQUE SUB-CATEGORIES
<StringArray>
[      'Large Cap',       'Small Cap',            'Gilt',         'Mid Cap',
  'Short Duration',           'Value',          'Liquid',       'Index/ETF',
       'Flexi Cap',           'Index', 'Large & Mid Cap',            'ELSS']
Length: 12, dtype: str

UNIQUE RISK GRADES


KeyError: 'risk_grade'

In [3]:
print("Fund houses by fund count:")
print(fund_master['fund_house'].value_counts())

print("\nCategory breakdown:")
print(fund_master['category'].value_counts())

print("\nSub-category breakdown:")
print(fund_master['sub_category'].value_counts())

print("\nRisk grade breakdown:")
print(fund_master['risk_grade'].value_counts())

Fund houses by fund count:
fund_house
SBI Mutual Fund             5
HDFC Mutual Fund            5
ICICI Prudential MF         5
Nippon India MF             5
Kotak Mahindra MF           4
Axis Mutual Fund            4
Aditya Birla Sun Life MF    3
UTI Mutual Fund             3
Mirae Asset MF              3
DSP Mutual Fund             3
Name: count, dtype: int64

Category breakdown:
category
Equity    34
Debt       6
Name: count, dtype: int64

Sub-category breakdown:
sub_category
Large Cap          14
Mid Cap             7
Small Cap           6
Liquid              3
Gilt                2
Flexi Cap           2
Short Duration      1
Value               1
Index/ETF           1
Index               1
Large & Mid Cap     1
ELSS                1
Name: count, dtype: int64

Risk grade breakdown:


KeyError: 'risk_grade'

In [1]:
import pandas as pd

fund_master = pd.read_csv('../data/raw/01_fund_master.csv')
nav_history = pd.read_csv('../data/raw/02_nav_history.csv')

In [2]:
master_codes = set(fund_master['amfi_code'])
nav_codes = set(nav_history['amfi_code'])

# Codes in fund_master but missing from nav_history
missing_from_nav = master_codes - nav_codes

# Codes in nav_history but not in fund_master (unexpected/orphan codes)
missing_from_master = nav_codes - master_codes

print("Total funds in fund_master:", len(master_codes))
print("Total unique funds in nav_history:", len(nav_codes))
print("\nFund codes MISSING from nav_history:", len(missing_from_nav))
if missing_from_nav:
    print(fund_master[fund_master['amfi_code'].isin(missing_from_nav)][['amfi_code', 'scheme_name']])

print("\nCodes in nav_history but NOT in fund_master (orphans):", len(missing_from_master))
if missing_from_master:
    print(missing_from_master)

Total funds in fund_master: 40
Total unique funds in nav_history: 40

Fund codes MISSING from nav_history: 0

Codes in nav_history but NOT in fund_master (orphans): 0


In [3]:
nav_counts = nav_history.groupby('amfi_code').size().reset_index(name='nav_record_count')
nav_counts = nav_counts.merge(fund_master[['amfi_code', 'scheme_name']], on='amfi_code', how='left')

print(nav_counts.sort_values('nav_record_count').head(10))  # funds with fewest NAV records
print("\nNAV record count stats:")
print(nav_counts['nav_record_count'].describe())

   amfi_code  nav_record_count  \
0     100016              1150   
1     100025              1150   
2     100033              1150   
3     101206              1150   
4     101207              1150   
5     101208              1150   
6     102885              1150   
7     102886              1150   
8     102887              1150   
9     118632              1150   

                                         scheme_name  
0          HDFC Top 100 Fund - Regular Plan - Growth  
1       HDFC Short Term Debt Fund - Regular - Growth  
2  HDFC Mid-Cap Opportunities Fund - Regular - Gr...  
3      ABSL Frontline Equity Fund - Regular - Growth  
4             ABSL Small Cap Fund - Regular - Growth  
5                ABSL Liquid Fund - Regular - Growth  
6         UTI Nifty 50 Index Fund - Regular - Growth  
7                UTI Mid Cap Fund - Regular - Growth  
8              UTI Flexi Cap Fund - Regular - Growth  
9     Nippon India Large Cap Fund - Regular - Growth  

NAV record count st

In [ ]:
total_funds = len(master_codes)
matched_funds = len(master_codes & nav_codes)
match_rate = (matched_funds / total_funds) * 100

summary = f"""
DATA QUALITY SUMMARY — AMFI Code Validation
=============================================
Total funds in fund_master:         {total_funds}
Funds with matching NAV history:    {matched_funds} ({match_rate:.1f}%)
Funds missing from nav_history:     {len(missing_from_nav)}
Orphan codes in nav_history:        {len(missing_from_master)}

NAV records per fund — min: {nav_counts['nav_record_count'].min()}, 
max: {nav_counts['nav_record_count'].max()}, 
mean: {nav_counts['nav_record_count'].mean():.0f}

Conclusion: {"All fund_master codes have corresponding NAV history — no gaps." if len(missing_from_nav) == 0 else f"{len(missing_from_nav)} funds have no NAV history and need investigation before analysis."}
"""

print(summary)

# Save the summary to a text file in reports/
with open('../reports/data_quality_summary.txt', 'w') as f:
    f.write(summary)

print("Summary saved to reports/data_quality_summary.txt")